In [1]:
!pip install wandb -q

In [2]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: uxvan (uxvan-personal) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [3]:
import os
import shutil
import subprocess
!pip install uv
!git clone https://github.com/Uxvan/assignment1-basics.git
os.chdir('/content/assignment1-basics')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.0/27.0 MB 64.2 MB/s eta 0:00:00
Cloning into 'assignment1-basics'...
remote: Enumerating objects: 959, done.
remote: Counting objects: 100% (4/4), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 959 (delta 0), reused 0 (delta 0), pack-reused 955 (from 2)
Receiving objects: 100% (959/959), 19.67 MiB | 29.84 MiB/s, done.
Resolving deltas: 100% (612/612), done.


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
shutil.copy('/content/drive/MyDrive/tinystories_valid.bin', '/content/assignment1-basics/tinystories_valid.bin')
shutil.copy('/content/drive/MyDrive/tinystories_train.bin', '/content/assignment1-basics/tinystories_train.bin')

%cd /content/assignment1-basics

/content


In [9]:
runs = {
    'baseline':    [],
    '2layers':     ['--num_layers', '2'],
    'dmodel256':   ['--d_model', '256'],
    'lr_low':      ['--lr_max', '3e-4'],
    'lr_high':     ['--lr_max', '3e-3'],
    'nowarmup':    ['--warm_iters', '0'],
    'nowd':        ['--weight_decay', '0'],
}

base_cmd = [
    'python', '-m', 'cs336_basics.trainingScript',
    '--train_data', '/content/assignment1-basics/tinystories_train.bin',
    '--val_data', '/content/assignment1-basics/tinystories_valid.bin',
    '--total_iters', '1000',
    '--eval_interval', '100',
    '--eval_iters', '20',
    '--log_interval', '50',
    '--seed', '42',
    '--use_wandb',
    '--wandb_project', 'cs336_ablation',
]

results = {}
for name, extra_args in runs.items():
    ckpt_path = f'/content/assignment1-basics/checkpoints/{name}.pt'
    cmd = base_cmd + [
        '--run_name', name,
        '--checkpoint_path', ckpt_path,
    ] + extra_args

    print(f'\n=== running {name}: {" ".join(extra_args) if extra_args else "(baseline)"} ===')
    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode != 0:
        print(f'--- {name} FAILED ---')
        print(result.stderr[-2000:])   # 只打印最后2000字符，避免刷屏
    else:
        print(f'--- {name} 完成 ---')
        # 抓取最后几行日志，快速看一眼train/val loss
        print('\n'.join(result.stdout.strip().split('\n')[-5:]))

    results[name] = result


=== running baseline: (baseline) ===
--- baseline 完成 ---
iter 800: val loss 2.1852
iter 850: train loss 2.0902, lr 0.000084, elapsed 427.2s
iter 900: train loss 2.1160, lr 0.000038, elapsed 450.5s
iter 900: val loss 2.1360
iter 950: train loss 2.1483, lr 0.000010, elapsed 477.1s

=== running 2layers: --num_layers 2 ===
--- 2layers 完成 ---
iter 800: val loss 2.3204
iter 850: train loss 2.2253, lr 0.000084, elapsed 261.8s
iter 900: train loss 2.2449, lr 0.000038, elapsed 276.1s
iter 900: val loss 2.2717
iter 950: train loss 2.2877, lr 0.000010, elapsed 292.3s

=== running dmodel256: --d_model 256 ===
--- dmodel256 完成 ---
iter 800: val loss 2.4079
iter 850: train loss 2.3163, lr 0.000084, elapsed 278.4s
iter 900: train loss 2.3301, lr 0.000038, elapsed 293.6s
iter 900: val loss 2.3639
iter 950: train loss 2.3809, lr 0.000010, elapsed 310.9s

=== running lr_low: --lr_max 3e-4 ===
--- lr_low 完成 ---
iter 800: val loss 2.5825
iter 850: train loss 2.5118, lr 0.000025, elapsed 430.9s
iter 900: 

In [10]:
shutil.copytree('/content/assignment1-basics/checkpoints', '/content/drive/MyDrive/checkpoints',)


'/content/drive/MyDrive/checkpoints'

In [6]:
import subprocess

cmd = [
    'python', '-u', '-m', 'cs336_basics.trainingScript',
    '--train_data', '/content/assignment1-basics/tinystories_train.bin',
    '--val_data', '/content/assignment1-basics/tinystories_valid.bin',
    '--total_iters', '1000',
    '--eval_interval', '100',
    '--eval_iters', '20',
    '--log_interval', '50',
    '--seed', '42',
    '--use_wandb',
    '--wandb_project', 'cs336_ablation',
    '--run_name', 'norope',
    '--checkpoint_path', '/content/assignment1-basics/checkpoints/norope.pt',
    '--theta', 'None',
]

result = subprocess.run(cmd)
if result.returncode != 0:
    print(f'FAILED (returncode={result.returncode})')
else:
    print('完成')

完成
